# Challenge 3 — Data Science & AI/ML-Model Evaluation, Benchmarking & Selection Research & Consulting Hub

**Student: Fabiana Rotella Campanari**

Content from **Lectures 01 to 09**, up to nested cross-validation (*nested CV*).

Run each cell and **write your response** in the indicated field. You **do not** need to write code — the goal is to interpret and compare the results. The `.csv` files are in this folder.

> **Rule for this challenge:** every statement that mentions a value must **bring the value**, copied from your cell output. Numbers change from notebook to notebook.

Some cells run multiple searches and can take one or two minutes. Wait for `[*]` to become a number before moving to the next.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from scipy.stats import uniform, loguniform, randint
from sklearn.model_selection import (train_test_split, cross_val_score, cross_val_predict,
                                     StratifiedKFold, GridSearchCV, RandomizedSearchCV)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score)

def logistica():
    """Logistic Regression inside a Pipeline (Lecture 05)."""
    return Pipeline([("esc", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=5000))])

def arvore():
    return DecisionTreeClassifier(max_depth=2, random_state=0)

def floresta():
    return RandomForestClassifier(n_estimators=50, random_state=0, n_jobs=-1)

print("environment ready")

environment ready


## Exercise 1 — Grid and Random Search with the same budget *(1.5 points)*

Two hyperparameters of the forest, 16 measurements for each path: a 4 × 4 grid and a `RandomizedSearchCV` with `n_iter = 16`. Then, the same random search repeated with 6 different seeds.

**(a)** Which path gave the best score, and by how much? Provide both values.

**(b)** How many **distinct** values of `max_features` did each path test? Connect this number to the difference in score.

**(c)** Look at the 6 seeds: how much does the random search result fluctuate? Cite the mean, minimum, maximum, and the fraction that scored above the grid. Can we declare a winner by comparing one number to another?

**(d)** If you added a third hyperparameter with 4 values, how many fits would each path cost? Show both calculations.

In [ ]:
df = pd.read_csv("q_sorteio.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
cv = StratifiedKFold(5, shuffle=True, random_state=0)

grade = {"max_features": [0.2, 0.4, 0.6, 0.8],
         "min_samples_leaf": [1, 5, 10, 20]}
g = GridSearchCV(floresta(), grade, cv=cv, scoring="accuracy", n_jobs=-1).fit(X, y)
print("grid  : 16 measurements | best", round(g.best_score_, 4), "|", g.best_params_)

dist = {"max_features": uniform(0.1, 0.8),
        "min_samples_leaf": randint(1, 21)}
r = RandomizedSearchCV(floresta(), dist, n_iter=16, cv=cv, scoring="accuracy",
                       random_state=0, n_jobs=-1).fit(X, y)
print("random: 16 measurements | best", round(r.best_score_, 4), "|",
      {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.best_params_.items()})

print("\ndistinct values of max_features tested:")
print("  grid  :", len(set(grade["max_features"])))
print("  random:", len(set(np.round(r.cv_results_["param_max_features"].data.astype(float), 8))))

notas = np.array([RandomizedSearchCV(floresta(), dist, n_iter=16, cv=cv,
                                     scoring="accuracy", random_state=s,
                                     n_jobs=-1).fit(X, y).best_score_
                  for s in range(6)])
print(f"\n6 seeds: mean {notas.mean():.4f} | min {notas.min():.4f}"
      f" | max {notas.max():.4f}")
print(f"fraction above grid ({g.best_score_:.4f}): {np.mean(notas >= g.best_score_):.0%}")

grid  : 16 measurements | best 0.6355 | {'max_features': 0.6, 'min_samples_leaf': 20}
random: 16 measurements | best 0.6355 | {'max_features': np.float64(0.5164), 'min_samples_leaf': 20}

distinct values of max_features tested:
  grid  : 4
  random: 16

6 seeds: mean 0.6335 | min 0.6235 | max 0.6474
fraction above grid (0.6355): 33%


**Response:**

**(a)** Both methods achieved exactly the same result! The accuracy obtained by the grid search was \( 0.6355 \) and the accuracy obtained by the random search was also exactly \( 0.6355 \). The absolute difference between the scores of both paths is \( 0.0000 \).

<br><br>

**(b)** The grid search evaluated only \( 4 \) distinct values for the `max_features` parameter (repeating combinations within its fixed space), whereas the random search evaluated \( 16 \) completely distinct values for `max_features`. Although the random search covered a parameter sample space four times larger in a well-distributed manner, the final score ended up tied at \( 0.6355 \) because the crucial influence on performance was tied to the limiting criterion `min_samples_leaf = 20` (which was found in both).

<br><br>

**(c)** The random search presents considerable fluctuations depending on the random seed adopted: the average accuracy observed across the 6 seeds was \( 0.6335 \), presenting a minimum score of \( 0.6235 \) and a maximum score of \( 0.6474 \). The fraction of seeds that outperformed or matched the grid score of \( 0.6355 \) was only \( 33\% \) (only two seeds out of six in total). With these numbers, it is evident that it is not correct to label random search as the definitive winner, since, depending on the initial sampling seed, it runs a considerable risk of performing below the grid search.

<br><br><br>

**(d)** If we added another parameter with \( 4 \) possible options:

The cost of the grid search would suffer a Cartesian product impact, costing:

<br>

$$ 4 \times 4 \times 4 \times 5 = 320 $$

<br>

model training and evaluation rounds.

The cost of the random search would remain strictly identical, totaling:

<br>

$$ 16 \times 5 = 80 $$

<br>

training rounds, since the upper limit `n_iter = 16` is strictly defined and controlled by the analyst.

──────────────────────── ⋆⋅☆⋅⋆ ────────────────────────

## Exercise 2 — Choosing and measuring on the same set *(1.5 points)*

The dataset `q_escolha.csv` is split into train and test. The first block scans seven values of `C` **measuring each on the test set** and keeps the best. The second block chooses `C` via cross-validation **within the training set** and touches the test set only once, at the very end.

**(a)** Which `C` and which score come out of each of the two paths? Give the values and the difference between the two final scores.

**(b)** Why is the score of the first path not the true performance of the model? Use the size of the test set in your explanation.

**(c)** What is the rule that the second path respects, and when can the test set be touched?

In [ ]:
df = pd.read_csv("q_escolha.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y,
                                          random_state=0)
print("train:", len(y_tr), "| test:", len(y_te))

valores = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

# PATH 1 — choosing C by looking at the TEST set
no_teste = {}
for C in valores:
    m = logistica().set_params(clf__C=C).fit(X_tr, y_tr)
    no_teste[C] = m.score(X_te, y_te)
print("\nscores ON TEST, by C:")
for C, s in no_teste.items():
    print(f"  C = {C:<7} -> {s:.4f}")
melhor_C = max(no_teste, key=no_teste.get)
print(f"best looking at test: C = {melhor_C} with {no_teste[melhor_C]:.4f}")

# PATH 2 — choosing via CV on training set and measuring only once
cv = StratifiedKFold(5, shuffle=True, random_state=0)
g = GridSearchCV(logistica(), {"clf__C": valores}, cv=cv, n_jobs=-1).fit(X_tr, y_tr)
honesta = logistica().set_params(clf__C=g.best_params_["clf__C"]).fit(X_tr, y_tr).score(X_te, y_te)
C_cv = g.best_params_["clf__C"]
print(f"\nchosen by CV on train: C = {C_cv} (CV score {g.best_score_:.4f})")
print(f"score on test, measured once: {honesta:.4f}")
print(f"difference to path 1: {no_teste[melhor_C] - honesta:+.4f}")

train: 106 | test: 46

scores ON TEST, by C:
  C = 0.001   -> 0.6739
  C = 0.01    -> 0.6522
  C = 0.1     -> 0.6304
  C = 1       -> 0.6087
  C = 10      -> 0.6304
  C = 100     -> 0.6304
  C = 1000    -> 0.6304
best looking at test: C = 0.001 with 0.6739

chosen by CV on train: C = 0.1 (CV score 0.6887)
score on test, measured once: 0.6304
difference to path 1: +0.0435


**Response:**

**(a)** In the first path, where the hyperparameter was chosen by directly monitoring performance on the test set, the best value obtained was \( C = 0.001 \) with a corresponding test accuracy of \( 0.6739 \). In the second path (with blinded choice of \( C \) via cross-validation on train), the selected value was \( C = 0.1 \) (which obtained \( 0.6887 \) of average validation accuracy) and resulted in a real final test accuracy of \( 0.6304 \). The difference between the test accuracies obtained in the two paths is \( +0.0435 \) in favor of the first approach.

<br><br>

**(b)** The score of \( 0.6739 \) in the first scenario is completely illusory and biased because the test set was actively used in choosing the estimator's configuration (resulting in *data leakage*). In a very small test set containing only \( 46 \) data samples, small statistical variations in correctness due to hyperparameter selection can artificially inflate metrics, masking the classifier's true ability to generalize to unseen data.

<br><br>

**(c)** The second method strictly adheres to the fundamental principle of data shielding, preventing information from leaking outside of training. Under this standard paradigm, the test set should only be queried once at the end of the experimental workflow to estimate the final generalization accuracy of the model with already fixed hyperparameters.

──────────────────────── ⋆⋅☆⋅⋆ ────────────────────────

## Exercise 4 — The grid and the edge of the range *(2.0 points)*

A `GridSearchCV` scans the `C` of logistic regression in a narrow range. The code prints the entire table of `cv_results_` and then the same search with the range expanded.

**(a)** Which `C` won in the first grid, and what was the score? Where is this value within the range that was scanned?

**(b)** What does a winner on the **edge** of the range indicate? Tell what you would do next.

**(c)** After expanding the range, which `C` won and what was the score? Was the first grid right or wrong? Explain.

**(d)** How many model fits did each of the two grids cost? Show the calculation.

In [ ]:
df = pd.read_csv("q_metrica.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
print("rows:", len(y), "| positives:", int(y.sum()), f"({y.mean():.1%})")

cv = StratifiedKFold(5, shuffle=True, random_state=0)
modelos = {"logistic": logistica(), "tree": arvore()}

for nome, m in modelos.items():
    p = cross_val_predict(m, X, y, cv=cv)
    print(f"\n{nome}")
    print(f"  accuracy : {accuracy_score(y, p):.4f}")
    print(f"  precision: {precision_score(y, p, zero_division=0):.4f}")
    print(f"  recall   : {recall_score(y, p, zero_division=0):.4f}")
    print(f"  F1       : {f1_score(y, p, zero_division=0):.4f}")
    print(f"  AP       : {cross_val_score(m, X, y, cv=cv, scoring='average_precision').mean():.4f}")

zero = np.zeros_like(y)
print(f"\nmodel that NEVER says 1 -> accuracy {accuracy_score(y, zero):.4f}"
      f" | F1 {f1_score(y, zero, zero_division=0):.4f}")

rows: 502 | positives: 38 (7.6%)

logistic
  accuracy : 0.9223
  precision: 0.4286
  recall   : 0.0789
  F1       : 0.1333
  AP       : 0.3399

tree
  accuracy : 0.9203
  precision: 0.4286
  recall   : 0.1579
  F1       : 0.2308
  AP       : 0.2008

model that NEVER says 1 -> accuracy 0.9243 | F1 0.0000


**Response:**

**(a)** Evaluating solely by the **accuracy** metric, the logistic regression wins the comparison, reaching \( 0.9223 \) compared to \( 0.9203 \) observed for the shallow decision tree. However, under the **F1-score** evaluation, the decision tree wins the matchup with \( 0.2308 \) compared to the low value of \( 0.1333 \) obtained by the logistic regression.

<br>

**(b)** Yes, the accuracy obtained by the baseline static classifier (which only predicts the majority class 0) is \( 0.9243 \), a value slightly superior to both the logistic regression accuracy (\( 0.9223 \)) and the decision tree accuracy (\( 0.9203 \)). In scenarios where the data is highly imbalanced — as is the case here, with only \( 38 \) positive instances out of \( 502 \) samples (only \( 7.6\% \) positives) —, accuracy severely masks classifier performance, because blindly predicting the majority class guarantees a high score without any actual detection of the positive class of interest.

<br>

**(c)** F1-score harmonically quantifies the relationship between Precision and Recall, penalizing models that ignore a class. Logistic regression achieved a high accuracy of \( 0.9223 \) because it almost never predicted the positive class (class 1), which resulted in a precision of \( 0.4286 \) coupled with an extremely low recall of only \( 0.0789 \) (detecting very few positives). This lack of sensitivity dragged its final F1-score down to just \( 0.1333 \).

<br>

**(d)** I would deliver the **logistic regression** model, justifying the choice using the **Average Precision (AP)** metric, where logistic regression reached \( 0.3399 \) compared to \( 0.2008 \) obtained by the decision tree. AP is the most robust metric for imbalanced datasets because it analyzes the ordering performance of the model across multiple decision thresholds (precision-recall curve), without being constrained to the default fixed threshold of \( 0.5 \).

──────────────────────── ⋆⋅☆⋅⋆ ────────────────────────


## Exercise 4 — The grid and the edge of the range *(2.0 points)*

A `GridSearchCV` scans the `C` of logistic regression in a narrow range. The code prints the entire table of `cv_results_` and then the same search with the range expanded.

**(a)** Which `C` won in the first grid, and what was the score? Where is this value within the range that was scanned?

**(b)** What does a winner on the **edge** of the range indicate? Tell what you would do next.

**(c)** After expanding the range, which `C` won and what was the score? Was the first grid right or wrong? Explain.

**(d)** How many model fits did each of the two grids cost? Show the calculation.

In [ ]:
df = pd.read_csv("q_grade.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
cv = StratifiedKFold(5, shuffle=True, random_state=0)

def busca_grade(valores):
    g = GridSearchCV(logistica(), {"clf__C": valores}, cv=cv,
                     scoring="accuracy", n_jobs=-1).fit(X, y)
    t = pd.DataFrame({"C": valores,
                      "mean": g.cv_results_["mean_test_score"].round(4),
                      "std": g.cv_results_["std_test_score"].round(4)})
    return g, t

faixa1 = [0.0001, 0.001, 0.01, 0.1]
g1, t1 = busca_grade(faixa1)
print("GRID 1 — range from 0.0001 to 0.1")
print(t1.to_string(index=False))
print("best C:", g1.best_params_["clf__C"], "| score:", round(g1.best_score_, 4))

faixa2 = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000]
g2, t2 = busca_grade(faixa2)
print("\nGRID 2 — range expanded up to 1000")
print(t2.to_string(index=False))
print("best C:", g2.best_params_["clf__C"], "| score:", round(g2.best_score_, 4))

GRID 1 — range from 0.0001 to 0.1
     C   mean    std
0.0001 0.8255 0.1083
0.0010 0.9117 0.0308
0.0100 0.9177 0.0246
0.1000 0.9359 0.0262
best C: 0.1 | score: 0.9359

GRID 2 — range expanded up to 1000
        C   mean    std
   0.0001 0.8255 0.1083
   0.0010 0.9117 0.0308
   0.0100 0.9177 0.0246
   0.1000 0.9359 0.0262
   1.0000 0.9696 0.0234
  10.0000 0.9786 0.0155
 100.0000 0.9939 0.0075
1000.0000 0.9908 0.0123
best C: 100 | score: 0.9939


**Response:**

**(a)** In the first grid, the winning hyperparameter was the limit value of \( C = 0.1 \), achieving an average accuracy of \( 0.9359 \). This value is located on the extreme right edge of the initial range searched, which was restricted to the interval \( [0.0001, 0.001, 0.01, 0.1] \).

<br><br>

**(b)** When the best hyperparameter selected is located exactly on the edge of the search interval, there is a strong indication that even more extreme values in that direction might yield superior performance. The correct next step is to expand the search grid, creating a wider range of tests in that direction.

<br><br>

**(c)** After expanding the search to include higher orders of magnitude, the optimal value obtained was \( C = 100.0 \), raising the average accuracy to \( 0.9939 \). Therefore, we conclude that the first grid search parameterization was conceptually wrong and sub-optimized, as the initial constraint prevented the classifier from exploring regions of better fit.

<br><br><br>

**(d)** The first search routine cost a total of \( 20 \) model trainings, resulting from:

<br>

$$ 4 \text{ values of } C \times 5 \text{ folds} = 20 $$

<br>

The second extended routine cost a total of \( 40 \) model trainings, resulting from:

<br>

$$ 8 \text{ values of } C \times 5 \text{ folds} = 40 $$

──────────────────────── ⋆⋅☆⋅⋆ ────────────────────────

## Exercise 5 — The three numbers of the same search *(2.0 points)*

Three numbers come out of the same search: the `best_score_`, the **nested CV**, and the score on the **held-out test set**. The code produces all three and displays the outer folds of the nested CV, one by one.

**(a)** Provide the three numbers. Which is the largest?

**(b)** Which of them **cannot** be reported as the model's performance, and why? (The answer is a matter of principle, not which one happened to be larger.)

**(c)** Look at the outer folds of the nested CV, one by one, and the standard deviation. What does this standard deviation say about the difference between the three numbers?

**(d)** What would you write in the final report, in one sentence, using the numbers from your notebook?

In [ ]:
df = pd.read_csv("q_nested.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y,
                                          random_state=0)
print("train:", len(y_tr), "| test:", len(y_te), "| positives in test:", int(y_te.sum()))

dentro = StratifiedKFold(5, shuffle=True, random_state=0)
fora   = StratifiedKFold(5, shuffle=True, random_state=1)
espaco = {"clf__C": loguniform(0.001, 1000)}
busca = RandomizedSearchCV(logistica(), espaco, n_iter=20, cv=dentro,
                           scoring="accuracy", random_state=0, n_jobs=-1)

nested = cross_val_score(busca, X_tr, y_tr, cv=fora, scoring="accuracy")
ajustada = busca.fit(X_tr, y_tr)
teste = ajustada.score(X_te, y_te)

print(f"\n1) best_score_   : {ajustada.best_score_:.4f}")
print(f"2) nested CV     : {nested.mean():.4f} +- {nested.std():.4f}")
print(f"   by fold       : {np.round(nested, 4)}")
print(f"3) held-out test : {teste:.4f}")

train: 140 | test: 60 | positives in test: 23

1) best_score_   : 0.7214
2) nested CV     : 0.7357 +- 0.0892
   by fold       : [0.7857 0.5714 0.7857 0.7143 0.8214]
3) held-out test : 0.6000


**Response:**

**(a)** The three collected values were:
1. Best internal accuracy (`best_score_`): \( 0.7214 \)
2. Average of nested cross-validation (`nested CV`): \( 0.7357 \)
3. Accuracy on the held-out test set: \( 0.6000 \)

The highest value among the three is the nested cross-validation mean (`nested CV`), which registered \( 0.7357 \).

<br><br>

**(b)** The `best_score_` indicator (\( 0.7214 \)) must not be reported as the model's stable generalization capacity. This number represents the optimized maximum value across multiple hyperparameter tests in the inner search, carrying an optimistic selection bias that masks the true performance of the model on entirely unseen data.

<br><br>

**(c)** Analyzing the five folds of the outer loop of the nested cross-validation, we observe scores that vary widely between \( 0.5714 \) and \( 0.8214 \) (specifically: \( 0.7857 \), \( 0.5714 \), \( 0.7857 \), \( 0.7143 \), and \( 0.8214 \)), resulting in a high standard deviation of \( 0.0892 \). This sharp oscillation shows that small sample sizes generate severe evaluation instability, which perfectly explains why the accuracy of the control test set (\( 0.6000 \)) fell below expectations, as it lies well within the normal statistical fluctuation range of the model.

<br><br>

**(d)** "According to the estimate obtained via nested cross-validation, the expected performance for our classifier is \( 0.7357 \pm 0.0892 \) in accuracy, which is statistically compatible with the score of \( 0.6000 \) recorded in our evaluation using the isolated test set."

──────────────────────── ⋆⋅☆⋅⋆ ────────────────────────

## Exercise 6 — Podium or plateau? *(1.5 points)*

A search scans the `C` of logistic regression across 12 values. The code prints the full podium — average and standard deviation between folds for each position.

**(a)** What is the difference between 1st and 5th place? And what is the standard deviation across folds for 1st place? Give both values and compare them.

**(b)** Is this a podium or a plateau? Justify using the numbers.

**(c)** Which configuration would you deliver, and why? (Hint: among tied configurations, there is a better criterion than "came in first".)

In [ ]:
df = pd.read_csv("q_plato.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
cv = StratifiedKFold(5, shuffle=True, random_state=0)

valores = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300]
g = GridSearchCV(logistica(), {"clf__C": valores}, cv=cv,
                 scoring="accuracy", n_jobs=-1).fit(X, y)
tab = pd.DataFrame({"C": valores,
                    "mean": g.cv_results_["mean_test_score"],
                    "std": g.cv_results_["std_test_score"]})
tab = tab.sort_values("mean", ascending=False).reset_index(drop=True)
tab.index = tab.index + 1
print(tab.round(4).to_string())

dif = tab["mean"].iloc[0] - tab["mean"].iloc[4]
print(f"\ndifference 1st - 5th: {dif:.4f}")
print(f"standard deviation of 1st: {tab['std'].iloc[0]:.4f}")

          C    mean     std
1     0.001  0.5688  0.0364
2     0.003  0.5625  0.0395
3     0.010  0.5562  0.0459
4     0.030  0.5500  0.0468
5     0.100  0.5438  0.0508
6     0.300  0.5375  0.0459
7     1.000  0.5375  0.0459
8     3.000  0.5375  0.0459
9    10.000  0.5375  0.0459
10   30.000  0.5375  0.0459
11  100.000  0.5375  0.0459
12  300.000  0.5375  0.0459

difference 1st - 5th: 0.0250
standard deviation of 1st: 0.0364


Response:

**(a)** The absolute difference in mean accuracy obtained between the first place in the table (\( C = 0.001 \) with a mean score of \( 0.5688 \)) and fifth place (\( C = 0.1 \) with a mean score of \( 0.5438 \)) is only \( 0.0250 \). Meanwhile, the standard deviation observed across the validation folds of the top model is \( 0.0364 \). Comparing them, the difference between the mean scores of the top positions (\( 0.0250 \)) is considerably lower than the average variation captured inside the data partitions (\( 0.0364 \)).

<br><br>

**(b)** This result is clearly a **plateau**. Since the performance standard deviation across the folds of the leading model (\( 0.0364 \)) is larger than the variation in mean score observed between the 1st and 5th positions in the rankings (\( 0.0250 \)), it is impossible to state with statistical confidence that the model in 1st place is truly superior to the subsequent models.

<br><br>

**(c)** I would deliver the configuration with lower complexity (meaning more regularization), choosing \( C = 0.001 \) (which scored an average of \( 0.5688 \) and a standard deviation of \( 0.0364 \)) or a nearby value within the statistical tie plateau, as the parsimony principle dictates that we should prefer simpler and more robust models to avoid overfitting to sample noise.

🛸๋*ੈ✩* 🔭✮☾𖤓.☘︎ ݁˖⁷⁷⁷ㅤ✮ ⋆ ˚｡𖦹 ⋆｡°✩ ✮ ⋆ ˚｡𖦹 ⋆｡°✩  *ੈ✩‧₊˚  🛸๋*ੈ✩* 🔭✮☾𖤓.☘︎ ݁˖⁷⁷⁷ㅤ✮ ⋆ ˚｡𖦹 ⋆｡°✩ ✮ ⋆ ˚｡𖦹 ⋆｡°✩  *ੈ✩‧₊˚  

### ⚡️